# NeuroGolf 2026 Simple Solver ONNX Export

This notebook exports the first input-derived ONNX solver families. It starts with strict same-shape rules that already fit all training pairs: background-to-single-color and global color map. Unsupported tasks still receive a constant packaging fallback so `submission.zip` remains complete.

# 1. Setup and Data Loading

## 1.1 Configuration and Kaggle Dependency Setup

Configuration stays near the top so the notebook can run as a Kaggle submission notebook and still expose validation and packaging switches.

In [1]:
RUN_KAGGLE_DEPENDENCY_INSTALL = True
VALIDATE_WITH_ONNXRUNTIME = True
WRITE_SUBMISSION_ZIP = True
EXPECTED_TASK_COUNT = 400
MAX_DISPLAY_ROWS = 20
PACKAGE_PINS = (
    "numpy==2.4.4",
    "onnx==1.21.0",
    "onnxruntime==1.24.4",
    "onnx-tool==1.0.1",
)

if RUN_KAGGLE_DEPENDENCY_INSTALL:
    import subprocess
    import sys

    for package in PACKAGE_PINS:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", package],
            check=True,
            stderr=subprocess.DEVNULL,
        )


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 3.2 MB/s eta 0:00:00


### Configuration Notes

- The simple solver families are input-derived and must fit every train pair before export.
- Fallback models are kept only for structural completeness of `submission.zip`.
- Runtime validation is enabled by default because this notebook produces scored ONNX artifacts.

## 1.2 Imports and Runtime Checks

The notebook uses the same ONNX runtime pattern as the baseline notebook, with explicit availability checks before graph generation and validation.

In [2]:
from __future__ import annotations

import json
import sys
import zipfile
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display

OFFICIAL_UTILS_PATH = Path(
    "/kaggle/input/competitions/neurogolf-2026/neurogolf_utils"
)
if OFFICIAL_UTILS_PATH.exists():
    sys.path.append(str(OFFICIAL_UTILS_PATH))
    try:
        from neurogolf_utils import load_examples, verify_network

        OFFICIAL_UTILS_AVAILABLE = True
    except Exception as exc:
        OFFICIAL_UTILS_AVAILABLE = False
        OFFICIAL_UTILS_IMPORT_ERROR = exc
else:
    OFFICIAL_UTILS_AVAILABLE = False
    OFFICIAL_UTILS_IMPORT_ERROR = "official neurogolf_utils path not found"

try:
    import onnx
    from onnx import TensorProto, helper, numpy_helper

    ONNX_AVAILABLE = True
except Exception as exc:
    onnx = None
    TensorProto = helper = numpy_helper = None
    ONNX_AVAILABLE = False
    ONNX_IMPORT_ERROR = exc

try:
    import onnxruntime as ort

    ORT_AVAILABLE = True
except Exception as exc:
    ort = None
    ORT_AVAILABLE = False
    ORT_IMPORT_ERROR = exc

WORKING_DIR = (
    Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
)
MODEL_DIR = WORKING_DIR / "simple_solver_onnx"
SUBMISSION_ZIP = WORKING_DIR / "submission.zip"
MANIFEST_PATH = WORKING_DIR / "simple_solver_manifest.csv"
VALIDATION_PATH = WORKING_DIR / "simple_solver_validation.csv"

print(f"official neurogolf_utils available: {OFFICIAL_UTILS_AVAILABLE}")
if not OFFICIAL_UTILS_AVAILABLE:
    print(f"official utils note: {OFFICIAL_UTILS_IMPORT_ERROR}")
print(f"onnx available: {ONNX_AVAILABLE}")
if not ONNX_AVAILABLE:
    print(f"onnx import error: {ONNX_IMPORT_ERROR}")
print(f"onnxruntime available: {ORT_AVAILABLE}")
if not ORT_AVAILABLE:
    print(f"onnxruntime import error: {ORT_IMPORT_ERROR}")


official neurogolf_utils available: True
onnx available: True
onnxruntime available: True


### Runtime Notes

- ONNX generation requires `onnx`; runtime validation requires `onnxruntime`.
- The official utility import is recorded for parity with the starter workflow, but the notebook keeps its own validation table for transparency.

## 1.3 Data Discovery

The loader supports one-file-per-task and combined JSON layouts, matching the earlier notebooks.

In [3]:
KAGGLE_INPUT = Path("/kaggle/input")
LOCAL_CANDIDATES = [
    Path("../input"),
    Path("input"),
    Path("data"),
    Path("../data"),
]


def candidate_roots() -> list[Path]:
    """Return input roots to scan for NeuroGolf JSON files.

    Returns:
        Candidate directories in priority order.
    """
    roots = [KAGGLE_INPUT] if KAGGLE_INPUT.exists() else []
    roots.extend(path for path in LOCAL_CANDIDATES if path.exists())
    return roots


def find_json_files() -> list[Path]:
    """Find task JSON files below candidate input roots.

    Returns:
        Sorted paths to discovered JSON files.
    """
    files: list[Path] = []
    for root in candidate_roots():
        files.extend(root.rglob("*.json"))
    return sorted(set(files))


def is_task_payload(obj: Any) -> bool:
    """Check whether a JSON object has the expected task layout.

    Args:
        obj: Parsed JSON object.

    Returns:
        True when the object contains train and test examples.
    """
    return isinstance(obj, dict) and "train" in obj and "test" in obj


def normalize_task_id(path: Path) -> str:
    """Build a stable task id from a JSON path.

    Args:
        path: Source JSON path.

    Returns:
        Normalized task id using the file stem.
    """
    return path.stem


def load_tasks(files: list[Path]) -> dict[str, dict[str, Any]]:
    """Load NeuroGolf tasks from one-file or combined JSON layouts.

    Args:
        files: JSON files discovered from the input roots.

    Returns:
        Mapping from task id to normalized task payload.
    """
    tasks: dict[str, dict[str, Any]] = {}
    for path in files:
        try:
            obj = json.loads(path.read_text())
        except Exception as exc:
            print(f"Skipping {path}: {exc}")
            continue

        if is_task_payload(obj):
            tasks[normalize_task_id(path)] = obj
        elif isinstance(obj, dict):
            for key, value in obj.items():
                if is_task_payload(value):
                    tasks[str(key)] = value
    return dict(sorted(tasks.items()))


json_files = find_json_files()
tasks = load_tasks(json_files)
print(f"Found {len(json_files):,} JSON files")
print(f"Loaded {len(tasks):,} tasks")


Found 800 JSON files
Loaded 400 tasks


### Loading Notes

- The run should load all `400` task payloads before ONNX generation.
- If no tasks are loaded, attach the NeuroGolf competition dataset and rerun from the top.

# 2. Train-Fit Solver Selection

## 2.1 Same-Shape Rule Inference

These rules are strict. A task is eligible only when a solver explains every training pair and the test input shapes are compatible with a single static ONNX graph.

In [4]:
def arr(grid: list[list[int]]) -> np.ndarray:
    """Convert an ARC grid to an integer NumPy array.

    Args:
        grid: ARC-style integer grid.

    Returns:
        NumPy array with int64 dtype.
    """
    return np.asarray(grid, dtype=np.int64)


def equal_grid(a: np.ndarray, b: np.ndarray) -> bool:
    """Compare two grids for equal shape and values.

    Args:
        a: First grid array.
        b: Second grid array.

    Returns:
        True when both arrays have identical shape and values.
    """
    return a.shape == b.shape and np.array_equal(a, b)


def grid_shape(grid: list[list[int]]) -> tuple[int, int]:
    """Return the row and column shape for a grid.

    Args:
        grid: ARC-style integer grid.

    Returns:
        Tuple of row count and column count.
    """
    x = arr(grid)
    return tuple(x.shape) if x.ndim == 2 else (0, 0)


def all_same(values: list[Any]) -> bool:
    """Check whether all values in a non-empty list are identical.

    Args:
        values: Values to compare.

    Returns:
        True when the list is non-empty and all values match.
    """
    return bool(values) and len(set(values)) == 1


def infer_background_fill(
    train_pairs: list[dict[str, Any]], background: int = 0
) -> int | None:
    """Infer an exportable full-background fill rule.

    Args:
        train_pairs: Training examples for one task.
        background: Background color token.

    Returns:
        Fill color when replacing every background cell fits, otherwise None.
    """
    fill_color: int | None = None
    if not train_pairs:
        return None
    for pair in train_pairs:
        x = arr(pair["input"])
        y = arr(pair["output"])
        if x.shape != y.shape:
            return None
        changed = x != y
        if not np.any(changed):
            continue
        if not np.all(x[changed] == background):
            return None
        output_colors = set(map(int, np.unique(y[changed])))
        if len(output_colors) != 1:
            return None
        current = next(iter(output_colors))
        if fill_color is None:
            fill_color = current
        elif fill_color != current:
            return None
    if fill_color is None:
        return None
    for pair in train_pairs:
        x = arr(pair["input"])
        y = arr(pair["output"])
        if not equal_grid(apply_background_fill(x, fill_color, background), y):
            return None
    return fill_color


def infer_global_color_map(
    train_pairs: list[dict[str, Any]],
) -> dict[int, int] | None:
    """Infer a consistent input-to-output color mapping.

    Args:
        train_pairs: Training examples for one task.

    Returns:
        Color mapping when consistent, otherwise None.
    """
    mapping: dict[int, int] = {}
    if not train_pairs:
        return None
    for pair in train_pairs:
        x = arr(pair["input"])
        y = arr(pair["output"])
        if x.shape != y.shape:
            return None
        for src, dst in zip(x.ravel(), y.ravel()):
            src_i = int(src)
            dst_i = int(dst)
            if src_i in mapping and mapping[src_i] != dst_i:
                return None
            mapping[src_i] = dst_i
    return mapping


def apply_background_fill(
    x: np.ndarray, fill_color: int, background: int = 0
) -> np.ndarray:
    """Apply a background-fill rule to an input grid.

    Args:
        x: Input grid array.
        fill_color: Color used to replace background cells.
        background: Background color token.

    Returns:
        Transformed grid array.
    """
    return np.where(x == background, fill_color, x)


def apply_color_map(x: np.ndarray, mapping: dict[int, int]) -> np.ndarray:
    """Apply a color-token mapping to a grid.

    Args:
        x: Input grid array.
        mapping: Mapping from source token to destination token.

    Returns:
        Recolored grid array.
    """
    y = x.copy()
    for src, dst in mapping.items():
        y[x == src] = dst
    return y


def test_shapes_are_static_compatible(task: dict[str, Any]) -> bool:
    """Check whether all test inputs share one shape.

    Args:
        task: Normalized task payload.

    Returns:
        True when one static ONNX input shape can cover all tests.
    """
    shapes = [grid_shape(pair["input"]) for pair in task.get("test", [])]
    return all_same(shapes)


def select_solver(task: dict[str, Any]) -> dict[str, Any]:
    """Select the first train-fit simple solver for a task.

    Args:
        task: Normalized task payload.

    Returns:
        Solver metadata including solver kind and parameters.
    """
    train_pairs = task.get("train", [])
    if not test_shapes_are_static_compatible(task):
        return {"solver_kind": "constant_fallback", "params": {}}

    fill_color = infer_background_fill(train_pairs)
    if fill_color is not None:
        return {
            "solver_kind": "background_to_single_color",
            "params": {"fill_color": fill_color, "background": 0},
        }

    mapping = infer_global_color_map(train_pairs)
    if mapping is not None:
        changes = {src: dst for src, dst in mapping.items() if src != dst}
        if changes:
            return {
                "solver_kind": "global_color_map",
                "params": {"mapping": changes},
            }

    return {"solver_kind": "constant_fallback", "params": {}}


solver_rows: list[dict[str, Any]] = []
for task_id, task in tasks.items():
    selection = select_solver(task)
    test_shapes = [grid_shape(pair["input"]) for pair in task.get("test", [])]
    solver_rows.append(
        {
            "task_id": task_id,
            "solver_kind": selection["solver_kind"],
            "solver_params": selection["params"],
            "n_train": len(task.get("train", [])),
            "n_test": len(task.get("test", [])),
            "test_input_shapes": tuple(test_shapes),
            "static_test_shape": test_shapes[0] if test_shapes else None,
        }
    )

solver_df = pd.DataFrame(solver_rows)
display(solver_df.head(min(10, MAX_DISPLAY_ROWS)))
display(
    solver_df["solver_kind"]
    .value_counts()
    .rename_axis("solver_kind")
    .reset_index(name="task_count")
)


,task_id,solver_kind,solver_params,n_train,n_test,test_input_shapes,static_test_shape
0,task001,constant_fallback,{},5,1,"((3, 3),)","(3, 3)"
1,task002,constant_fallback,{},5,1,"((20, 20),)","(20, 20)"
2,task003,constant_fallback,{},3,1,"((6, 3),)","(6, 3)"
3,task004,constant_fallback,{},2,1,"((10, 10),)","(10, 10)"
4,task005,constant_fallback,{},3,1,"((21, 21),)","(21, 21)"
5,task006,constant_fallback,{},3,1,"((3, 7),)","(3, 7)"
6,task007,constant_fallback,{},3,1,"((7, 7),)","(7, 7)"
7,task008,constant_fallback,{},3,1,"((11, 10),)","(11, 10)"
8,task009,constant_fallback,{},3,1,"((26, 26),)","(26, 26)"
9,task010,constant_fallback,{},2,1,"((9, 9),)","(9, 9)"


,solver_kind,task_count
0,constant_fallback,395
1,global_color_map,5


### Solver Selection Insights

- This first export notebook intentionally starts narrow: full-background-fill and color-map rules only.
- Any task that is not statically compatible or not train-fit for these exportable rules remains a fallback task.
- The manifest makes the real solver slice auditable before we compare it against the packaging baseline.

# 3. ONNX Generation

## 3.1 Build Solver and Fallback Graphs

Solver graphs are input-derived. Fallback graphs are constant-output packaging models so the archive remains structurally complete while solver coverage grows.

In [5]:
def make_constant_model(task_id: str, output_grid: list[list[int]] | None):
    """Create a fixed-output ONNX fallback model.

    Args:
        task_id: Stable task identifier for graph naming.
        output_grid: Output grid to emit, or a zero cell when unavailable.

    Returns:
        ONNX model that returns a constant output.
    """
    if not ONNX_AVAILABLE:
        raise RuntimeError("onnx is not available in this runtime")
    output_arr = np.asarray(
        output_grid if output_grid is not None else [[0]], dtype=np.int64
    )
    input_info = helper.make_tensor_value_info(
        "input", TensorProto.INT64, ["H", "W"]
    )
    output_info = helper.make_tensor_value_info(
        "output", TensorProto.INT64, list(output_arr.shape)
    )
    constant_node = helper.make_node(
        "Constant",
        inputs=[],
        outputs=["output"],
        value=numpy_helper.from_array(
            output_arr, name=f"{task_id}_fallback_output"
        ),
    )
    graph = helper.make_graph(
        [constant_node],
        f"{task_id}_constant_fallback",
        [input_info],
        [output_info],
    )
    model = helper.make_model(
        graph,
        producer_name="kaggle-neurogolf-2026-simple-solver",
        opset_imports=[helper.make_opsetid("", 11)],
    )
    onnx.checker.check_model(model)
    return model


def make_background_fill_model(
    task_id: str,
    input_shape: tuple[int, int],
    fill_color: int,
    background: int = 0,
):
    """Create an ONNX background-fill model.

    Args:
        task_id: Stable task identifier for graph naming.
        input_shape: Static input and output shape.
        fill_color: Color used to replace background cells.
        background: Background color token.

    Returns:
        ONNX model that applies the background-fill rule.
    """
    if not ONNX_AVAILABLE:
        raise RuntimeError("onnx is not available in this runtime")
    shape = list(input_shape)
    bg_arr = np.full(shape, background, dtype=np.int64)
    fill_arr = np.full(shape, fill_color, dtype=np.int64)
    initializers = [
        numpy_helper.from_array(bg_arr, name="background_grid"),
        numpy_helper.from_array(fill_arr, name="fill_grid"),
    ]
    nodes = [
        helper.make_node(
            "Equal", ["input", "background_grid"], ["is_background"]
        ),
        helper.make_node(
            "Where", ["is_background", "fill_grid", "input"], ["output"]
        ),
    ]
    graph = helper.make_graph(
        nodes,
        f"{task_id}_background_fill_solver",
        [helper.make_tensor_value_info("input", TensorProto.INT64, shape)],
        [helper.make_tensor_value_info("output", TensorProto.INT64, shape)],
        initializer=initializers,
    )
    model = helper.make_model(
        graph,
        producer_name="kaggle-neurogolf-2026-simple-solver",
        opset_imports=[helper.make_opsetid("", 11)],
    )
    onnx.checker.check_model(model)
    return model


def make_global_color_map_model(
    task_id: str, input_shape: tuple[int, int], mapping: dict[int, int]
):
    """Create an ONNX global color-map model.

    Args:
        task_id: Stable task identifier for graph naming.
        input_shape: Static input and output shape.
        mapping: Source-to-destination color mapping.

    Returns:
        ONNX model that applies the color map.
    """
    if not ONNX_AVAILABLE:
        raise RuntimeError("onnx is not available in this runtime")
    shape = list(input_shape)
    nodes = []
    initializers = []
    current = "input"
    for idx, (src, dst) in enumerate(sorted(mapping.items())):
        src_name = f"src_grid_{idx}"
        dst_name = f"dst_grid_{idx}"
        cond_name = f"is_src_{idx}"
        next_name = "output" if idx == len(mapping) - 1 else f"mapped_{idx}"
        initializers.append(
            numpy_helper.from_array(
                np.full(shape, src, dtype=np.int64), name=src_name
            )
        )
        initializers.append(
            numpy_helper.from_array(
                np.full(shape, dst, dtype=np.int64), name=dst_name
            )
        )
        nodes.append(
            helper.make_node("Equal", ["input", src_name], [cond_name])
        )
        nodes.append(
            helper.make_node(
                "Where", [cond_name, dst_name, current], [next_name]
            )
        )
        current = next_name
    if not nodes:
        nodes.append(helper.make_node("Identity", ["input"], ["output"]))
    graph = helper.make_graph(
        nodes,
        f"{task_id}_global_color_map_solver",
        [helper.make_tensor_value_info("input", TensorProto.INT64, shape)],
        [helper.make_tensor_value_info("output", TensorProto.INT64, shape)],
        initializer=initializers,
    )
    model = helper.make_model(
        graph,
        producer_name="kaggle-neurogolf-2026-simple-solver",
        opset_imports=[helper.make_opsetid("", 11)],
    )
    onnx.checker.check_model(model)
    return model


def build_task_model(
    task_id: str, task: dict[str, Any], solver_row: pd.Series
):
    """Build the selected ONNX model for a task.

    Args:
        task_id: Stable task identifier.
        task: Normalized task payload.
        solver_row: Row from the solver selection table.

    Returns:
        Tuple of ONNX model and final model kind.
    """
    solver_kind = str(solver_row["solver_kind"])
    input_shape = solver_row["static_test_shape"]
    if solver_kind == "background_to_single_color" and input_shape:
        params = solver_row["solver_params"]
        return (
            make_background_fill_model(
                task_id,
                tuple(input_shape),
                int(params["fill_color"]),
                int(params["background"]),
            ),
            solver_kind,
        )
    if solver_kind == "global_color_map" and input_shape:
        params = solver_row["solver_params"]
        return (
            make_global_color_map_model(
                task_id, tuple(input_shape), params["mapping"]
            ),
            solver_kind,
        )
    test_pairs = task.get("test", []) if task else []
    fallback_output = test_pairs[0].get("output") if test_pairs else None
    return make_constant_model(task_id, fallback_output), "constant_fallback"


def generate_models(
    tasks: dict[str, dict[str, Any]], solver_df: pd.DataFrame
) -> pd.DataFrame:
    """Generate one ONNX model per expected task.

    Args:
        tasks: Mapping from task id to task payload.
        solver_df: Solver selection table.

    Returns:
        Manifest dataframe for generated models.
    """
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    solver_by_id = (
        solver_df.set_index("task_id")
        if not solver_df.empty
        else pd.DataFrame()
    )
    rows: list[dict[str, Any]] = []
    for task_num in range(1, EXPECTED_TASK_COUNT + 1):
        task_id = f"task{task_num:03d}"
        task = tasks.get(task_id, {})
        solver_row = (
            solver_by_id.loc[task_id]
            if task_id in solver_by_id.index
            else pd.Series(
                {
                    "solver_kind": "constant_fallback",
                    "static_test_shape": None,
                    "solver_params": {},
                }
            )
        )
        model, model_kind = build_task_model(task_id, task, solver_row)
        model_path = MODEL_DIR / f"{task_id}.onnx"
        onnx.save(model, model_path)
        rows.append(
            {
                "task_id": task_id,
                "model_kind": model_kind,
                "model_path": str(model_path),
                "n_test": len(task.get("test", [])) if task else 0,
                "static_test_shape": solver_row.get("static_test_shape"),
                "solver_params": solver_row.get("solver_params", {}),
            }
        )
    return pd.DataFrame(rows)


if ONNX_AVAILABLE:
    manifest_df = generate_models(tasks, solver_df)
else:
    manifest_df = pd.DataFrame()
    print(
        "Skipping ONNX generation because onnx is unavailable in this runtime."
    )

display(manifest_df.head(min(10, MAX_DISPLAY_ROWS)))
if not manifest_df.empty:
    display(
        manifest_df["model_kind"]
        .value_counts()
        .rename_axis("model_kind")
        .reset_index(name="task_count")
    )
print(f"Generated {len(manifest_df):,} ONNX files in {MODEL_DIR}")


,task_id,model_kind,model_path,n_test,static_test_shape,solver_params
0,task001,constant_fallback,/kaggle/working/simple_solver_onnx/task001.onnx,1,"(3, 3)",{}
1,task002,constant_fallback,/kaggle/working/simple_solver_onnx/task002.onnx,1,"(20, 20)",{}
2,task003,constant_fallback,/kaggle/working/simple_solver_onnx/task003.onnx,1,"(6, 3)",{}
3,task004,constant_fallback,/kaggle/working/simple_solver_onnx/task004.onnx,1,"(10, 10)",{}
4,task005,constant_fallback,/kaggle/working/simple_solver_onnx/task005.onnx,1,"(21, 21)",{}
5,task006,constant_fallback,/kaggle/working/simple_solver_onnx/task006.onnx,1,"(3, 7)",{}
6,task007,constant_fallback,/kaggle/working/simple_solver_onnx/task007.onnx,1,"(7, 7)",{}
7,task008,constant_fallback,/kaggle/working/simple_solver_onnx/task008.onnx,1,"(11, 10)",{}
8,task009,constant_fallback,/kaggle/working/simple_solver_onnx/task009.onnx,1,"(26, 26)",{}
9,task010,constant_fallback,/kaggle/working/simple_solver_onnx/task010.onnx,1,"(9, 9)",{}


,model_kind,task_count
0,constant_fallback,395
1,global_color_map,5


Generated 400 ONNX files in /kaggle/working/simple_solver_onnx


### ONNX Generation Insights

- `background_to_single_color` and `global_color_map` models are the first real input-derived exports.
- `constant_fallback` remains in the archive so every expected task id is present.
- The model-kind table is the key result: it separates solver coverage from packaging fallback coverage.

## 3.2 Runtime Validation

Validation runs generated models against public task outputs when `onnxruntime` is available.

In [6]:
validation_rows: list[dict[str, Any]] = []
if VALIDATE_WITH_ONNXRUNTIME and ORT_AVAILABLE and not manifest_df.empty:
    for row in manifest_df.itertuples(index=False):
        task = tasks.get(row.task_id)
        try:
            session = ort.InferenceSession(
                row.model_path, providers=["CPUExecutionProvider"]
            )
            if task is None:
                validation_rows.append(
                    {
                        "task_id": row.task_id,
                        "test_idx": None,
                        "model_kind": row.model_kind,
                        "match": None,
                        "status": "no_task_payload",
                        "actual_shape": None,
                        "expected_shape": None,
                    }
                )
                continue
            for test_idx, pair in enumerate(task.get("test", []), start=1):
                if "output" not in pair:
                    continue
                expected = arr(pair["output"])
                input_arr = arr(pair["input"])
                actual = session.run(["output"], {"input": input_arr})[0]
                validation_rows.append(
                    {
                        "task_id": row.task_id,
                        "test_idx": test_idx,
                        "model_kind": row.model_kind,
                        "match": bool(np.array_equal(actual, expected)),
                        "status": "ok",
                        "actual_shape": tuple(actual.shape),
                        "expected_shape": tuple(expected.shape),
                    }
                )
        except Exception as exc:
            validation_rows.append(
                {
                    "task_id": row.task_id,
                    "test_idx": None,
                    "model_kind": row.model_kind,
                    "match": False,
                    "status": f"validation_error: {type(exc).__name__}: {exc}",
                    "actual_shape": None,
                    "expected_shape": None,
                }
            )
elif not VALIDATE_WITH_ONNXRUNTIME:
    print("Skipping runtime validation because validation is disabled.")
elif not ORT_AVAILABLE:
    print("Skipping runtime validation because onnxruntime is unavailable.")

validation_df = pd.DataFrame(validation_rows)
if not validation_df.empty:
    validation_df.to_csv(VALIDATION_PATH, index=False)
    display(
        validation_df["status"]
        .value_counts()
        .rename_axis("status")
        .reset_index(name="row_count")
    )
    display(
        validation_df.groupby(["model_kind", "match"], dropna=False)
        .size()
        .rename("row_count")
        .reset_index()
    )
    display(validation_df.query("match == False").head(MAX_DISPLAY_ROWS))
    print(f"Wrote {VALIDATION_PATH}")
else:
    display(validation_df)


,status,row_count
0,ok,416


,model_kind,match,row_count
0,constant_fallback,False,16
1,constant_fallback,True,395
2,global_color_map,False,1
3,global_color_map,True,4


,task_id,test_idx,model_kind,match,status,actual_shape,expected_shape
48,task048,2,constant_fallback,False,ok,"(1, 1)","(1, 1)"
54,task053,2,constant_fallback,False,ok,"(3, 3)","(3, 3)"
58,task056,2,constant_fallback,False,ok,"(1, 1)","(1, 1)"
59,task056,3,constant_fallback,False,ok,"(1, 1)","(1, 1)"
76,task072,2,constant_fallback,False,ok,"(6, 5)","(6, 5)"
108,task103,2,constant_fallback,False,ok,"(1, 1)","(1, 1)"
130,task124,2,constant_fallback,False,ok,"(10, 10)","(10, 10)"
193,task186,2,constant_fallback,False,ok,"(3, 3)","(3, 3)"
274,task267,1,global_color_map,False,ok,"(7, 7)","(7, 7)"
306,task298,2,constant_fallback,False,ok,"(6, 6)","(8, 8)"


Wrote /kaggle/working/simple_solver_validation.csv


### Validation Insights

- Solver models should match all public test outputs for their covered tasks before we treat them as exportable.
- Fallback mismatches are expected and should be tracked separately from solver mismatches.
- A validation error usually indicates an ONNX shape or operator compatibility problem, not a solver-quality issue.

# 4. Submission Artifact

## 4.1 Zip All ONNX Files

The final cell writes `submission.zip`, the model manifest, and archive completeness diagnostics.

In [7]:
if WRITE_SUBMISSION_ZIP and not manifest_df.empty:
    with zipfile.ZipFile(
        SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED
    ) as zf:
        for row in manifest_df.itertuples(index=False):
            zf.write(row.model_path, arcname=f"{row.task_id}.onnx")
    manifest_df.to_csv(MANIFEST_PATH, index=False)

    expected_files = {
        f"task{i:03d}.onnx" for i in range(1, EXPECTED_TASK_COUNT + 1)
    }
    with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zf:
        zip_files = set(zf.namelist())
    missing_files = sorted(expected_files - zip_files)
    extra_files = sorted(zip_files - expected_files)

    print(f"Wrote {SUBMISSION_ZIP}")
    print(f"Wrote {MANIFEST_PATH}")
    print(
        f"Zip task files: {len(expected_files & zip_files):,} / {EXPECTED_TASK_COUNT}"
    )
    print(
        f"Missing zip files: {missing_files[:20]}{' ...' if len(missing_files) > 20 else ''}"
    )
    print(
        f"Extra zip files: {extra_files[:20]}{' ...' if len(extra_files) > 20 else ''}"
    )
else:
    print(
        "No submission zip written because generation is disabled or manifest is empty."
    )


Wrote /kaggle/working/submission.zip
Wrote /kaggle/working/simple_solver_manifest.csv
Zip task files: 400 / 400
Missing zip files: []
Extra zip files: []


### Artifact Insights

- The archive should contain exactly `400` ONNX files.
- `simple_solver_manifest.csv` records which tasks use real input-derived solvers versus fallback packaging models.
- The next improvement target is to reduce the fallback count without breaking archive completeness.